# Visual Computing: Object Tracking and Motion Analysis
 In this coursework, you will implement various object tracking algorithms and motion analysis techniques using computer vision. The goal is to understand and apply different tracking approaches, analyse their performance, and evaluate their effectiveness under real-world conditions. This coursework is worth 7% of the total marks for the unit.

### Change Detection Using GMM [10%]
#### Objective:
Implement Change Detection using a Gaussian Mixture Model (GMM) for object tracking from scratch. Follow the algorithm outlined in the provided lecture slide for modelling pixel changes.
cv2.createBackgroundSubtractorMOG2, is not allowed.
#### Implementation Details:
Model the image as a mixture of Gaussians and classify each pixel as foreground or background. This will be done based on pixel probability distributions, as explained in the lecture. Update the Gaussian distributions per pixel to decide whether a pixel belongs to the background or foreground. Evaluate the effectiveness of the model in classifying moving objects. You can use cv2.VideoCapture(), cv2.cvtColor(),cv2.COLOR_BGR2GRAY(), cv2.waitKey().


In [ ]:
# Gaussian Mixture Model (GMM) for Change Detection
# -------------------------------------------------
# Instructions: Below, I provide a code structure as a guidance to get you started. Of course you can use your own code structure to achieve the objective. Your marks will NOT be deducted if you use your own code structure :) 
# Complete the sections marked with '### ENTER YOUR CODE HERE ###' to implement 
# the Gaussian Mixture Model for background subtraction and change detection.
'''
useful resources:
https://moodle.bath.ac.uk/mod/forum/discuss.php?d=534501
https://ieeexplore.ieee.org/document/4723224
'''
import cv2
import numpy as np

# Initialize the GMMBackgroundSubtractor Class
class GMMBackgroundSubtractor:
    def __init__(self, frame_shape, num_gaussians= 4, learning_rate=0.001, threshold=2.5):
        """
        Initialize Gaussian parameters: means, variances, and weights.

        Args:
            frame_shape (tuple): Shape of the input frame (height, width).
            num_gaussians (int): Number of Gaussian models per pixel.
            learning_rate (float): Rate at which the model updates.
            threshold (float): Threshold for matching a pixel to a Gaussian.
        """
        # diff
        self.num_gaussians = num_gaussians
        self.learning_rate = learning_rate
        self.threshold = threshold
        
        # diff
        # Initialize the means, variances, and weights for each Gaussian
        self.means = np.random.randint(0, 256, (frame_shape[0], frame_shape[1], self.num_gaussians)).astype(np.float32)
        self.variances = np.full((frame_shape[0], frame_shape[1], self.num_gaussians), 10**2, dtype = np.float32)
        self.weights = np.full((frame_shape[0], frame_shape[1], self.num_gaussians), 1 / self.num_gaussians, dtype=np.float32)

        # Normalise the weights
        self.weights /= np.sum(self.weights, axis=2, keepdims=True)  
        self.weights = np.clip(self.weights, 0.01, 1)

        
        
    def apply(self, frame):
        """
        Apply GMM to detect foreground objects.

        Args:
            frame (ndarray): Input video frame.

        Returns:
            foreground_mask (ndarray): Binary mask indicating foreground pixels.
        """
        # Convert frame to grayscale if it isn't already
        if len(frame.shape) == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        frame = frame.astype(np.float32)
        
        #diff # Expand frame to match gaussian dimensions
        pixel_values = np.repeat(frame[:, :, np.newaxis], self.num_gaussians, axis=2)
        
        # Calculate the absolute difference between the frame and Gaussian means
        abs_diff = np.abs(frame[:, : , np.newaxis] - self.means) # increase the dimension of the existing array

        # Check which pixels match any of the Gaussians
        matched = abs_diff < (self.threshold * np.sqrt(self.variances))
        
        # Create an empty foreground mask
        foreground_mask = np.ones(frame.shape, dtype=np.uint8) 

        # Step 1: Update matched Gaussians (means, variances, weights)
        
        matched_gaussian = np.argmax(matched, axis=2) 

        for i in range(self.num_gaussians):
            match = (matched_gaussian == i)
            self.means[match, i] = (1 - self.learning_rate) * self.means[match, i] + self.learning_rate * frame[match]
            self.variances[match, i] = (1 - self.learning_rate) * self.variances[match, i] + self.learning_rate * (frame[match] - self.means[match, i])**2
            self.weights[match, i] += self.learning_rate    

        # Reduce the weights of unmatched Gaussians
        self.weights[~matched] *= (1 - self.learning_rate * 2)
            
        # Normalize weights so they sum to 1
        self.weights /= np.sum(self.weights, axis=2, keepdims=True)
        self.weights = np.clip(self.weights, 0.01, 1)

        # Step 2: Classify Foreground
        best_fit = np.any(matched, axis=2)  
        foreground_mask[best_fit] = 0  
        
        return foreground_mask.astype(np.uint8) * 255

# ------------------- Main Code to Run the GMM ------------------- #

# Load Video
video_path = 'GMM_input-cars.mp4'
cap = cv2.VideoCapture(video_path)


# Read the first frame to get the frame shape
ret, frame = cap.read()

if not ret:
    print("Error: Could not read the video file")
    cap.release()
    exit()

# Initialize GMM Subtractor
gmm_subtractor = GMMBackgroundSubtractor(frame.shape[:2], num_gaussians=3, learning_rate=0.001, threshold=2.5)

# Process the video frame by frame
### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###
while True:  
    ret, frame = cap.read()
    if not ret:
        break
    
    # Apply GMM background subtraction to get the foreground mask
    foreground_mask = gmm_subtractor.apply(frame)
    
    # Clean the noise 
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 1))

    # Morphological Operations
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_CLOSE, kernel)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_CLOSE, kernel)
    foreground_mask = cv2.dilate(foreground_mask, kernel, iterations=2)
    #foreground_mask = cv2.GaussianBlur(foreground_mask, (5, 5), 0) #tried the guassian blur, makes the result worse
    #_, foreground_mask = cv2.threshold(foreground_mask, 127, 255, cv2.THRESH_BINARY)

    # Display the foreground mask
    cv2.imshow('Input', frame)
    cv2.imshow('Foreground Mask', foreground_mask)
    
    # Quit the video (press 'q')
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources and close windows
cap.release()
cv2.destroyAllWindows()

### Custom Lucas-Kanade (with OpenCV GMM) [10%]
#### Objective
In this task, students will be using Lucas-Kanade Optical Flow to track detected moving objects. You can use the OpenCV GMM cv2.createBackgroundSubtractorMOG2() for this section.  
#### Implementation Details:
Detect foreground objects using the OpenCV GMM, then extract feature points inside the foreground mask. Thereafter, implement and apply your custom Lucas-Kanade Optical Flow to track moving objects frame-by-frame. Visualise this via Motion vectors which are optical flow arrows showing direction and magnitude of object movement. You can use cv2.Sobel(), cv2.VideoCapture(), cv2.goodFeaturesToTrack(), cv2.cvtColor(),cv2.COLOR_BGR2GRAY(), cv2.waitKey(), and for visualization cv2.circle(),cv2.arrowedLine. 


In [ ]:
import cv2
import numpy as np

class LucasKanadeTracker:
    def __init__(self):
        # These are the settings for Shi-Tomasi corner detection
        self.feature_params = dict(
            maxCorners=100,     
            qualityLevel=0.3,   
            minDistance=7,      
            blockSize=7         
        )
        
        # Size of the window around each point we'll use to compute optical flow
        self.window_size = 15
        self.half_window = self.window_size // 2
        
        # Pyramidal LK settings
        self.max_level = 3
        self.scale_factor = 0.5  # Downscale factor 
        
        # LK iteration and convergence settings
        self.max_iterations = 10
        self.epsilon = 0.01 # Stop iterating if the flow update is smaller than this
        
        # Used to discard points that are too flat or untrackable
        self.min_eigenvalue = 0.01

    def detect_features(self, frame, mask):
        """
        Detects corner features (keypoints) in the current frame,
        but only within the foreground mask (moving objects).
        """
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return cv2.goodFeaturesToTrack(gray, mask=mask, **self.feature_params)

    def build_pyramid(self, image):
        """Builds an image pyramid by repeatedly downsampling.
        This is used to track motion from coarse to fine levels"""
        pyramid = [image.copy()]
        
        for level in range(1, self.max_level + 1):
            next_img = cv2.resize(pyramid[-1], 
                                  (int(pyramid[-1].shape[1] * self.scale_factor),
                                   int(pyramid[-1].shape[0] * self.scale_factor)),
                                  interpolation=cv2.INTER_LINEAR)
            pyramid.append(next_img)
            
        return pyramid

    def compute_flow_at_level(self, prev_gray, curr_gray, points, guess=None):
        """For each point, estimate how much it moved between two frames
        at the current resolution level"""
        if points is None or len(points) == 0:
            return np.array([]), np.array([])
        
        h, w = prev_gray.shape
        new_points = []
        status = []
        
        # Reduce noise in gradients
        prev_gray = cv2.GaussianBlur(prev_gray, (5, 5), 0)
        curr_gray = cv2.GaussianBlur(curr_gray, (5, 5), 0)
        
        # Calculate image gradients (Ix and Iy) fromo the previous frame
        Ix = cv2.Sobel(prev_gray, cv2.CV_64F, 1, 0, ksize=3)
        Iy = cv2.Sobel(prev_gray, cv2.CV_64F, 0, 1, ksize=3)
        
        for i, pt in enumerate(points):
            x, y = pt.ravel()
            x, y = int(x), int(y)
            
            # If there's a guess from the previous level, add it to the current point
            if guess is not None:
                x += guess[i][0]
                y += guess[i][1]
            
            # Skip points too close to the edge as we can't form a window around them
            if x < self.half_window or y < self.half_window or \
               x >= w - self.half_window or y >= h - self.half_window:
                new_points.append([[x, y]])
                status.append(0)
                continue
            
            # Define the window region around the point
            x_start, x_end = x - self.half_window, x + self.half_window + 1
            y_start, y_end = y - self.half_window, y + self.half_window + 1
            
            template = prev_gray[y_start:y_end, x_start:x_end]
            
            # Update the displacement iteratively
            dx, dy = 0.0, 0.0
            
            try:
                for _ in range(self.max_iterations):
                    nx, ny = x + dx, y + dy
                    nx_int, ny_int = int(nx), int(ny)
                    
                    # Make sure the new window is still within the image
                    if nx_int < self.half_window or ny_int < self.half_window or \
                       nx_int >= w - self.half_window or ny_int >= h - self.half_window:
                        break
                    
                    # Get current window position based on current flow estimate
                    curr_x_start = int(nx - self.half_window)
                    curr_x_end = int(nx + self.half_window + 1)
                    curr_y_start = int(ny - self.half_window)
                    curr_y_end = int(ny + self.half_window + 1)
                    
                    if curr_x_start < 0 or curr_y_start < 0 or \
                       curr_x_end > w or curr_y_end > h:
                        break
                    
                    target = curr_gray[curr_y_start:curr_y_end, curr_x_start:curr_x_end]
                    
                    # Calculate intensity difference between windows
                    It = target.astype(np.float64) - template.astype(np.float64)
                    
                    # Flatten gradients and form matrices
                    Ix_win = Ix[y_start:y_end, x_start:x_end].flatten()
                    Iy_win = Iy[y_start:y_end, x_start:x_end].flatten()
                    It_win = It.flatten()
                    
                    # Form the system of equations
                    A = np.vstack((Ix_win, Iy_win)).T
                    b = -It_win
                    
                    # Solving the flow: (A^T A) * [dx, dy] = A^T b
                    G = np.dot(A.T, A)
                    
                    # Check if the system is solvable
                    # Use eigenvalue analysis to determine if the point is trackable
                    eig_vals = np.linalg.eigvals(G)
                    min_eig = np.min(eig_vals)
                    
                    if min_eig < self.min_eigenvalue:
                        break # Not trackable
                        
                    # Calculate flow update
                    flow_update = np.linalg.solve(G, np.dot(A.T, b))
                    
                    dx += flow_update[0]
                    dy += flow_update[1]
                    
                    if np.abs(flow_update[0]) < self.epsilon and np.abs(flow_update[1]) < self.epsilon:
                        break
                
                # Final sanity check: ignore huge motions 
                flow_magnitude = np.sqrt(dx*dx + dy*dy)
                if flow_magnitude < 50:
                    new_points.append([[x + dx, y + dy]])
                    status.append(1)
                else:
                    new_points.append([[x, y]])
                    status.append(0)
                    
            except Exception as e:
                new_points.append([[x, y]])
                status.append(0)
        
        return np.array(new_points, dtype=np.float32), np.array(status, dtype=np.int32).reshape(-1)

    def calculate_optical_flow(self, prev_frame, curr_frame, prev_points):
        """
        Tracks points using pyramidal LK across levels
        """
        if prev_points is None or len(prev_points) == 0:
            return np.array([]), np.array([])
        
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        
        # Build pyramids for both frames
        prev_pyramid = self.build_pyramid(prev_gray)
        curr_pyramid = self.build_pyramid(curr_gray)
        
        # This will hold the flow estimate from coarse to fine levels
        initial_guess = np.zeros((len(prev_points), 2), dtype=np.float32)
        
        # Start with the coarsest level and work up to the finest level
        for level in range(self.max_level, -1, -1):
            level_scale = self.scale_factor ** level
            level_points = prev_points.copy()
            
            for i in range(len(level_points)):
                level_points[i][0][0] = prev_points[i][0][0] * level_scale
                level_points[i][0][1] = prev_points[i][0][1] * level_scale
            
            for i in range(len(level_points)):
                level_points[i][0][0] += initial_guess[i][0]
                level_points[i][0][1] += initial_guess[i][1]
                
            # Track features at this resolution level
            new_points, status = self.compute_flow_at_level(
                prev_pyramid[level], curr_pyramid[level], level_points)
            
            # Update flow for next level
            if level > 0:
                scale_to_next = 1.0 / self.scale_factor
                flow = np.zeros((len(prev_points), 2), dtype=np.float32)
                
                for i in range(len(prev_points)):
                    if status[i]:
                        # Calculate flow at this level
                        flow[i][0] = (new_points[i][0][0] - level_points[i][0][0])
                        flow[i][1] = (new_points[i][0][1] - level_points[i][0][1])
                        
                        # Scale up for next level
                        initial_guess[i][0] = flow[i][0] * scale_to_next
                        initial_guess[i][1] = flow[i][1] * scale_to_next
        
        # Final updated positions of all points
        final_points = np.zeros_like(prev_points)
        final_status = np.zeros(len(prev_points), dtype=np.int32)
        
        for i in range(len(prev_points)):
            x, y = prev_points[i][0]
            if status[i]:
                final_points[i][0][0] = x + initial_guess[i][0]
                final_points[i][0][1] = y + initial_guess[i][1]
                final_status[i] = 1
            else:
                final_points[i][0][0] = x
                final_points[i][0][1] = y
                final_status[i] = 0
        
        return final_points, final_status

    def draw_motion_vectors(self, frame, prev_points, new_points, status):
        """
        Draws motion vectors (arrows) showing where each feature moved.
        - Green arrow: movement direction from previous point to new point
        - Red circle: new point location
        """
        for i, (new, old) in enumerate(zip(new_points, prev_points)):
            if status[i]: 
                a, b = map(int, new.ravel())
                c, d = map(int, old.ravel())
                
                # Only draw significant movement
                displacement = np.sqrt((a-c)**2 + (b-d)**2)
                if displacement > 1.0:
                    cv2.arrowedLine(frame, (c, d), (a, b), (0, 255, 0), 2, tipLength=0.4)
                    cv2.circle(frame, (a, b), 3, (0, 0, 255), -1)
                    
        return frame


# Load Video
cap = cv2.VideoCapture('Lucas-Kanade-input_video.mp4') # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###

# Initialize GMM Background Subtractor with tuned parameters to isolate moving objects
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=50, varThreshold=25, detectShadows=False
)

# Prepare the background model for stability
for _ in range(30):
    ret, warmup_frame = cap.read()
    if not ret:
        break
    bg_subtractor.apply(warmup_frame, learningRate=0.5)

# Read the first frame from the input video
ret, prev_frame = cap.read()
if not ret:
    print("Error: Couldn't read video.")
    cap.release()
    exit()

# Initialize Lucas-Kanade Tracker
lk_tracker = LucasKanadeTracker()

# Apply background subtraction to find foreground in the first frame
foreground_mask = bg_subtractor.apply(prev_frame)
prev_points = lk_tracker.detect_features(prev_frame, foreground_mask)

# Initialize frame counter
frame_count = 0

# Process the video frame-by-frame
while cap.isOpened():
    ret, curr_frame = cap.read()
    if not ret:
        break
        
    frame_count += 1

    # Step 1: Apply background subtraction to get the foreground mask
    # This will show which parts of the frame are moving
    foreground_mask = bg_subtractor.apply(curr_frame)

    # Step 2: Clean the mask using Gaussian blur and thresholding
    # This will remove flickering and makes object shapes smoother
    foreground_mask = cv2.GaussianBlur(foreground_mask, (5, 5), 0)
    _, foreground_mask = cv2.threshold(foreground_mask, 127, 255, cv2.THRESH_BINARY)

    # Step 3: Morphological operations to remove noise and fill holes
    # Opening removes small noise, closing fills small gaps inside objects
    kernel = np.ones((3, 3), np.uint8)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_CLOSE, kernel)

    # Step 4: If we have points from the last frame, track them
    if prev_points is not None and len(prev_points) > 0:
        # Calculate optical flow
        new_points, status = lk_tracker.calculate_optical_flow(prev_frame, curr_frame, prev_points)

        # Draw the motion arrows and circles
        tracked_frame = lk_tracker.draw_motion_vectors(curr_frame.copy(), prev_points, new_points, status)

        # Show results
        cv2.imshow('Foreground Mask', foreground_mask)
        cv2.imshow('Optical Flow Tracking', tracked_frame)

        # Keep only the points that were successfully tracked
        valid_indices = np.where(status == 1)[0]
        if len(valid_indices) > 0:
            # Reshape valid points for the next iteration
            prev_points = new_points[valid_indices].reshape(-1, 1, 2)
        else:
            prev_points = None
    else:
        cv2.imshow('Optical Flow Tracking', curr_frame)
        cv2.imshow('Foreground Mask', foreground_mask)
    
    # Step 5: Periodically or if tracking fails, detect new features
    # This keeps the tracker fresh and prevents losing all points over time
    if frame_count % 10 == 0 or prev_points is None or (prev_points is not None and len(prev_points) < 15):
        prev_points = lk_tracker.detect_features(curr_frame, foreground_mask)
        
    # Step 6: Update the previous frame for next iteration
    prev_frame = curr_frame.copy()

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

# Reference:
# This was adapted based on the work done by Khushboo Agarwal
# Available at: https://github.com/khushboo-agarwal/Optical-Flow/blob/master/PA2_2.py

### Template Matching for Object Tracking [10%] 
#### Objective:
Students will implement template matching using a weighted histogram matching technique from scratch to track a target object across multiple frames in a video. Experiment with NCC to obtain the results.
#### Implementation Details:
User manually selects a target object in the first frame. Apply template matching to locate the object in subsequent frames. Experiment with the similarity metric:  NCC (Normalized Cross-Correlation) between template & search region. Draw a bounding box around the detected object in each frame (The best match is marked with a rectangle in each frame). You can use cv2.matchTemplate(),cv2.selectROI(),cv2.minMaxLoc (),cv2.TM_CCORR_NORMED (),cv2.COLOR_BGR2GRAY (), cv2.VideoCapture(), cv2.cvtColor(), cv2.waitKey(),cv2.normalize().



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def compute_epanechnikov_kernel():
    r = 8
    x = np.arange(-7.5, 8.5)
    y = np.arange(-7.5, 8.5)

    X, Y = np.meshgrid(x, y)

    distances = X**2 + Y**2

    kernel = np.maximum(0, 1 - (distances / r**2))

    return kernel

ep_kernel = compute_epanechnikov_kernel()

class TemplateMatcher:
    def __init__(self):
        pass

    def select_template(self, frame):
        """
        Allow the user to manually select the target object in the first frame.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.selectROI() to select the region of interest (ROI).

        r = cv2.selectROI("select", frame)

        return frame[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])]

    def match_template(self, frame, template, temp_hist):
        """
        Apply template matching using Normalized Cross-Correlation (NCC).
        """
        ### ENTER YOUR CODE HERE ###

        fr_width = frame.shape[1]
        fr_height = frame.shape[0]
        temp_width = template.shape[1]
        temp_height = template.shape[0]

        results = np.zeros((frame.shape[0] - template.shape[0], frame.shape[1] - template.shape[1]))
        
        for i in range(frame.shape[0] - template.shape[0]):
            for j in range(frame.shape[1] - template.shape[1]):
                histogram = np.zeros([256])
                for x in range(template.shape[0]):
                    for y in range(template.shape[1]):
                        value = int(frame[x+i, y+j])
                        histogram[value] += 1

                histogram = histogram.reshape(16, 16) / 255
                histogram = np.matmul(histogram, ep_kernel).astype(np.float32)

                result = cv2.matchTemplate(histogram, temp_hist, cv2.TM_CCORR_NORMED)
                results[i, j] = result[0][0]

        minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(results, None)

        return maxLoc

    def draw_bounding_box(self, frame, top_left, template_size):
        """
        Draw a bounding box around the detected object.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.rectangle() to draw the box.

        return cv2.rectangle(frame, top_left, (top_left[0] + template_size[1], top_left[1] + template_size[0]), (0,0,0), 2)

# Load Video
cap = cv2.VideoCapture('believecropevenlower.mp4')  # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###
success = 1
count = 0

tm = TemplateMatcher()

success, frame = cap.read()

template = tm.select_template(frame)

temp_grey = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

temp_width = template.shape[1]
temp_height = template.shape[0]

temp_hist = np.zeros([256])
for x in range(temp_grey.shape[0]):
    for y in range(temp_grey.shape[1]):
        value = int(temp_grey[x, y])
        temp_hist[value] += 1

temp_hist = temp_hist.reshape(16, 16) / 255
temp_hist = np.matmul(temp_hist, ep_kernel).astype(np.float32)

video = cv2.VideoWriter("output.mp4", cv2.VideoWriter_fourcc(*"XVID"), 30, (frame.shape[1], frame.shape[0]))

while success:
    success, img = cap.read()

    if not success:
        break

    img_grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    topleft = tm.match_template(img_grey, temp_grey, temp_hist)

    img = tm.draw_bounding_box(img, topleft, (template.shape[0], template.shape[1]))

    video.write(img)

cap.release()
video.release()
cv2.destroyAllWindows()

### Improving Template Matching for Object Tracking [5% + 10%]
#### Objective:
After implementing template matching, students should explore potential improvements.  The key challenges with template matching are: 1. Scale changes 2. Rotation Changes 3. Brightness/ contrast changes and Occlusions.
#### Implementation Details:
[A] Multi-Scale Template Matching (Handling Scale Changes)
Instead of using a fixed-size template, track multiple scales: Resize the template to 3 different scales (of your choice) before matching. Match at different pyramid levels using cv2.pyrDown() and cv2.pyrUp(). You can use prebuilt functions: cv2.matchTemplate(),cv2.minMaxLoc(),cv2.resize().

[B] Rotation-Invariant Matching
Rotate the template at 3 different angles (of your choice)  and match each rotated version. You can use cv2.getRotationMatrix2D(),cv2.warpAffine().

[C] Using Feature-Based Matching Instead of Pixel-Based Matching
Template matching relies on raw pixel values, making it sensitive to changes. Instead of comparing pixel intensities, extract SIFT features and follow the algorithm for tracking by feature detection discussed in the lecture. You can use cv2.SIFT_create(), sift.detectAndCompute,cv2.BFMatcher(), cv2.drawMatches() function here to compare. 


In [ ]:
import cv2
import numpy as np

def compute_epanechnikov_kernel():
    r = 8
    x = np.arange(-7.5, 8.5)
    y = np.arange(-7.5, 8.5)

    X, Y = np.meshgrid(x, y)

    distances = X**2 + Y**2

    kernel = np.maximum(0, 1 - (distances / r**2))

    return kernel

ep_kernel = compute_epanechnikov_kernel()

class ImprovedTemplateMatcher:
    def __init__(self):
        pass

    def select_template(self, frame):
        """
        Allow the user to manually select the target object in the first frame.
        """
        r = cv2.selectROI("select", frame)
        return frame[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])]

    def match_template(self, frame, template, temp_hist):
        """
        Apply template matching using Normalized Cross-Correlation (NCC).
        """
        ### ENTER YOUR CODE HERE ###

        fr_width = frame.shape[1]
        fr_height = frame.shape[0]
        temp_width = template.shape[1]
        temp_height = template.shape[0]

        results = np.zeros((frame.shape[0] - template.shape[0], frame.shape[1] - template.shape[1]))
        
        for i in range(frame.shape[0] - template.shape[0]):
            for j in range(frame.shape[1] - template.shape[1]):
                histogram = np.zeros([256])
                for x in range(template.shape[0]):
                    for y in range(template.shape[1]):
                        value = int(frame[x+i, y+j])
                        histogram[value] += 1

                histogram = histogram.reshape(16, 16) / 255
                histogram = np.matmul(histogram, ep_kernel).astype(np.float32)

                result = cv2.matchTemplate(histogram, temp_hist, cv2.TM_CCORR_NORMED)
                results[i, j] = result[0][0]

        minVal, maxVal, minLoc, maxLoc = cv2.minMaxLoc(results, None)

        return maxVal, maxLoc
    
    def multi_scale_matching(self, frame, template, temp_hist):
        """
        Perform multi-scale template matching to handle scale changes.
        """
        ### ENTER YOUR CODE HERE ###
        # Resize the template to different scales 
        scales = [0.8, 1.0, 1.2]
        pyramid_levels = [0, 1, 2]
        best_max_val = -np.inf
        for scale in scales:
            resized_template = cv2.resize(template, (0, 0), fx=scale, fy=scale)
            for level in pyramid_levels:
                resized_frame = self.get_pyramid_frame(frame, level)
                if (resized_template.shape[0] > resized_frame.shape[0] or
                    resized_template.shape[1] > resized_frame.shape[1]):
                    continue
                    
                max_val, max_loc = self.match_template(resized_frame, resized_template, temp_hist)

                if max_val > best_max_val:
                    best_max_val = max_val
                    best_scale = scale
                    best_match_info = {
                        'location': max_loc,
                        'scale': best_scale,
                        'level': level,
                        'match_value': max_val,
                        'template_size': resized_template.shape
                    }
                    
        level = best_match_info['level']
        scale_factor = 1 / 2 if level == 1 else (2 if level == 2 else 1)
        
        top_left = (
            int(best_match_info['location'][0] / scale_factor),
            int(best_match_info['location'][1] / scale_factor)
        )

        template_width, template_height = best_match_info['template_size'][:2]
        template_size = top_left[0] + int(template_height * best_scale), top_left[1] + int(template_width * best_scale)
        print(top_left, template_size)

        return top_left, template_size

    def get_pyramid_frame(self, frame, level):
        if level == 0:
            return frame.copy()
        elif level == 1:
            return cv2.pyrDown(frame)
        elif level == 2:
            return cv2.pyrUp(frame)

    def rotation_invariant_matching(self, frame, template, temp_hist):
        """
        Perform rotation-invariant template matching.
        """
        ### ENTER YOUR CODE HERE ###
        # Rotate the template at different angles         
        angles = [0.0, 25.0, 335.0]
        template_height, template_width = template.shape[:2]

        best_loc = None
        best_max_val = -np.inf
        best_angle = None
        center = (template_height/2, template_width/2) 
        
        for angle in angles:
            rotate_matrix = cv2.getRotationMatrix2D(center=center, angle=angle, scale = 1)
            rotated_template = cv2.warpAffine(template, rotate_matrix, dsize=(template_width, template_height))
            max_val, max_loc = self.match_template(resized_frame, rotated_template, temp_hist)

            if max_val > best_max_val:
                best_max_val = max_val
                best_loc = max_loc
                best_angle = angle
                rotated_template_height, rotated_template_width = rotated_template.shape[:2]
        
        top_left = best_loc
        bottom_right = (top_left[0] + rotated_template_width, top_left[1] + rotated_template_height)
        return top_left, bottom_right


    def tau(self,  template, prev_template):
        x, y, w, h = template
        x_prev, y_prev, w_prev, h_prev = prev_template
    
        dx = (x + w / 2) - (x_prev + w_prev / 2)
        dy = (y + h / 2) - (y_prev + h_prev / 2)
        dist = np.sqrt(dx**2 + dy**2)
    
        size_penalty = abs(w - w_prev) + abs(h - h_prev)
    
        return dist + 0.5 * size_penalty

    def construct_search_windows(self, frame, template):
        """
        Construct search windows by sliding the template across the entire frame.
        """
        windows = []
        template_height, template_width = template.shape[:2]
    
        # Define step size (stride) for sliding window
        stride = 5  # You can adjust this to be more or less granular
        for y in range(0, frame.shape[0] - template_height, stride):
            for x in range(0, frame.shape[1] - template_width, stride):
                window = (x, y, template_width, template_height)  # (x, y, width, height)
                windows.append(window)
        return windows
    
    def feature_based_matching(self, frame, template):
        """
        Use SIFT feature detection for robust template matching.
        """
        sift = cv2.SIFT_create()
        bf = cv2.BFMatcher()

        fea1, _ = sift.detectAndCompute(template, None)
        fea2, _ = sift.detectAndCompute(frame, None)

        obj_features = []
        bg_features = []
        
        for kp in fea1:  # fea1 = keypoints in template
            if x <= kp.pt[0] <= x+w and y <= kp.pt[1] <= y+h:
                obj_features.append(kp)
            else:
                bg_features.append(kp)
    
        matches = bf.knnMatch(des1, des2, k=2)

        
        good = []
        for m, n in matches:
            if m.distance/n.distance < 0.5:
                good.append(m)

        phi = len(good)
        best_score = -np.inf
        best_window = None
        search_windows = self.construct_search_windows(frame, template)
        for W in search_windows:  # Assume you have a list of search windows to evaluate
            # Compute τ(W) for the current search window
            tau_penalty = self.tau(best_window, W)
            
            # Calculate match score
            match_score = phi - tau_penalty  # Penalize windows with large deviations
            if match_score > best_score:
                best_score = match_score
                best_window = W
        print(W, W.shape)
        return W

    def draw_bounding_box(self, frame, top_left, template_size):
        """
        Draw bounding box around the detected object.
        """
        ### ENTER YOUR CODE HERE ###
        # Use cv2.rectangle() to draw the bounding box.
        # Window name in which image is displayed
        # template size, detected object.
        return cv2.rectangle(frame, top_left, template_size, color=(0,255,0), thickness=2)
        

# Load Video
cap = cv2.VideoCapture('/Users/tiennamnguyen/Desktop/1 - tracking, task 4/Tracking_model_inputs_outputs_extra_tests/4.Tracking by Feature Detection_Model_input_outputs/girl_input_video_cropped.mp4')  # Replace with your video file

### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###
success = 1
count = 0

tm = ImprovedTemplateMatcher()

success, frame = cap.read()

template = tm.select_template(frame)

temp_grey = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)

temp_width = template.shape[1]
temp_height = template.shape[0]

temp_hist = np.zeros([256])
for x in range(temp_grey.shape[0]):
    for y in range(temp_grey.shape[1]):
        value = int(temp_grey[x, y])
        temp_hist[value] += 1

temp_hist = temp_hist.reshape(16, 16) / 255
temp_hist = np.matmul(temp_hist, ep_kernel).astype(np.float32)

video = cv2.VideoWriter("output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 30, (frame.shape[1], frame.shape[0]))

while success:
    success, img = cap.read()

    if not success:
        break

    img_grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    topleft, bottomright = tm.feature_based_matching(img, template)

    img = tm.draw_bounding_box(img, topleft, bottomright)

    video.write(img)

cap.release()
video.release()
cv2.destroyAllWindows()